# Validation of logical resource estimates

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook validates the logical-qubit counts and non-Clifford-depth bounds derived in Appendix F of the paper. Each implemented circuit is recursively decomposed into the Clifford+$T$+$R_z$ gate set and compared with the corresponding analytical resource formula.

Logical-qubit counts are checked exactly. The depth formulas are upper bounds, so an implementation is accepted when its measured non-Clifford depth does not exceed the stated bound. Special parameter choices may produce a smaller depth.

**Table of contents**

1. [Import and validation helper](#import-and-validation-helper)
2. [Primitive gates](#primitive-gates)
3. [Multi-controlled NOT](#multi-controlled-not)
4. [Proposal preparation](#proposal-preparation)
5. [Fully phase-based arithmetic](#fully-phase-based-arithmetic)
6. [Hybrid fixed-point and phase arithmetic](#hybrid-arithmetic)
7. [Reflection and accept-path unitary](#reflection-and-accept-path)
8. [Validation notes](#validation-notes)

<a id="import-and-validation-helper"></a>
## Import and validation helper

The helper below checks the logical-qubit count and computes the non-Clifford depth using the explicit circuit decompositions in `monaqa2`.


In [1]:
import sys

if ".." not in sys.path:
    sys.path.append("..")  # Import the local package without an editable installation.

import numpy as np
from qiskit.circuit import QuantumCircuit
from monaqa2.qiskit.utils_qiskit import get_nc_depth


def ceil_log2(x: int) -> int:
    """Return ceil(log2(x)) for a positive integer."""
    if x < 1:
        raise ValueError("x must be positive.")
    return int(np.ceil(np.log2(x)))


def validate_resource(
    operation,
    expected_num_qubits: int,
    expected_depth: int,
    *,
    verbose: bool = False,
) -> None:
    """Validate an exact qubit count and an upper bound on non-Clifford depth."""
    name = getattr(operation, "name", operation.__class__.__name__)
    actual_num_qubits = operation.num_qubits

    if actual_num_qubits != expected_num_qubits:
        raise ValueError(
            f"{name} uses {actual_num_qubits} qubits; "
            f"expected {expected_num_qubits}."
        )

    if isinstance(operation, QuantumCircuit):
        circuit = operation
    else:
        circuit = QuantumCircuit(expected_num_qubits)
        circuit.append(operation, range(expected_num_qubits))

    actual_depth = get_nc_depth(circuit)
    if actual_depth > expected_depth:
        raise ValueError(
            f"{name} has non-Clifford depth {actual_depth}; "
            f"the analytical upper bound is {expected_depth}."
        )

    if verbose:
        print(
            f"{name}: qubits={actual_num_qubits}, "
            f"depth={actual_depth} <= {expected_depth}"
        )


<a id="primitive-gates"></a>
## Primitive gates

The following checks reproduce Table III of the paper.

| Gate | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{CCX}$ | $3$ | $3$ |
| $\mathrm{CR}_y(\theta)$ | $2$ | $2$ |
| $\mathrm{CCR}_y(\theta)$ | $3$ | $8$ |
| $G(\theta)$ | $2$ | $2$ |
| $\mathrm{C}G(\theta)$ | $3$ | $14$ |


### Direct validation

The primitive gates are instantiated with generic nonzero angles so that their explicit decompositions contain the expected non-Clifford layers.


In [2]:
from monaqa2.qiskit.primitives import (
    Ccx,
    Cry,
    Ccry,
    GivensRotation,
    ControlledGivensRotation,
)

validate_resource(Ccx(), 3, 3, verbose=True)
validate_resource(Cry(0.1), 2, 2, verbose=True)
validate_resource(Ccry(0.1), 3, 8, verbose=True)
validate_resource(GivensRotation(0.3, 0.7), 2, 2, verbose=True)
validate_resource(
    ControlledGivensRotation(0.3, 0.7),
    3,
    14,
    verbose=True,
)


ccx: qubits=3, depth=3 <= 3
cry: qubits=2, depth=2 <= 2
ccry: qubits=3, depth=8 <= 8
givens: qubits=2, depth=2 <= 2
c_givens: qubits=3, depth=14 <= 14


<a id="multi-controlled-not"></a>
## Multi-controlled NOT

The two-clean-ancilla construction `synth_mcx_2_clean_kg24` implements an $m$-controlled NOT using $m+3$ logical qubits. For $m\geq3$, its non-Clifford depth is upper-bounded by

$$
14\left\lceil\log_2m\right\rceil-10.
$$

The following cell checks the bound over consecutive and power-of-two control sizes.


In [3]:
from qiskit.synthesis.multi_controlled.mcx_synthesis import (
    synth_mcx_2_clean_kg24,
)

control_sizes = list(range(3, 16)) + [2**power for power in range(4, 16)]

for num_controls in control_sizes:
    expected_depth = 14 * ceil_log2(num_controls) - 10
    operation = synth_mcx_2_clean_kg24(num_controls)

    print(f"m={num_controls:5d} | ", end="")
    validate_resource(
        operation,
        expected_num_qubits=num_controls + 3,
        expected_depth=expected_depth,
        verbose=True,
    )


m=    3 | mcx_logn_depth: qubits=6, depth=11 <= 18
m=    4 | mcx_logn_depth: qubits=7, depth=18 <= 18
m=    5 | mcx_logn_depth: qubits=8, depth=22 <= 32
m=    6 | mcx_logn_depth: qubits=9, depth=27 <= 32
m=    7 | mcx_logn_depth: qubits=10, depth=27 <= 32
m=    8 | mcx_logn_depth: qubits=11, depth=28 <= 32
m=    9 | mcx_logn_depth: qubits=12, depth=30 <= 46
m=   10 | mcx_logn_depth: qubits=13, depth=32 <= 46
m=   11 | mcx_logn_depth: qubits=14, depth=36 <= 46
m=   12 | mcx_logn_depth: qubits=15, depth=36 <= 46
m=   13 | mcx_logn_depth: qubits=16, depth=40 <= 46
m=   14 | mcx_logn_depth: qubits=17, depth=42 <= 46
m=   15 | mcx_logn_depth: qubits=18, depth=44 <= 46
m=   16 | mcx_logn_depth: qubits=19, depth=46 <= 46
m=   32 | mcx_logn_depth: qubits=35, depth=56 <= 60
m=   64 | mcx_logn_depth: qubits=67, depth=74 <= 74
m=  128 | mcx_logn_depth: qubits=131, depth=84 <= 88
m=  256 | mcx_logn_depth: qubits=259, depth=102 <= 102
m=  512 | mcx_logn_depth: qubits=515, depth=110 <= 116
m= 1024 |

<a id="proposal-preparation"></a>
## Proposal preparation

These checks reproduce Table IV of the paper.

| Proposal block | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Uniform proposal | $2n$ | $0$ |
| Local proposal | $2n$ | $2\lceil\log_2n\rceil$ |
| Hamiltonian-simulation proposal | $2n$ | $r(n+3)$ |

The implementation class retains the internal name `ProposalQemc`; it represents the Hamiltonian-simulation proposal used in the paper. For even $n$, the implementation may use one fewer interaction-matching layer than the uniform bound $n$, but the validation is intentionally performed against the paper's bound $r(n+3)$.


In [4]:
from monaqa2.qiskit.proposal_uniform import ProposalUniform
from monaqa2.qiskit.proposal_local import ProposalLocal
from monaqa2.qiskit.proposal_qemc import ProposalQemc


def validate_proposal_resources(
    n: int,
    *,
    num_trotter_steps: int = 50,
) -> None:
    """Validate the three proposal rows in Table IV."""
    if n < 2:
        raise ValueError("Use n >= 2.")

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    gamma = np.ones(n)
    evolution_time = 1.0

    validate_resource(
        ProposalUniform(n),
        expected_num_qubits=2 * n,
        expected_depth=0,
    )
    validate_resource(
        ProposalLocal(n, k=1),
        expected_num_qubits=2 * n,
        expected_depth=2 * ceil_log2(n),
    )
    validate_resource(
        ProposalQemc(
            n,
            h,
            J,
            gamma,
            evolution_time,
            num_trotter_steps=num_trotter_steps,
        ),
        expected_num_qubits=2 * n,
        expected_depth=num_trotter_steps * (n + 3),
    )

    print(f"All proposal checks passed for n={n}.")


proposal_sizes = list(range(2, 16)) + [2**power for power in range(4, 8)]
for n in proposal_sizes:
    validate_proposal_resources(n)


All proposal checks passed for n=2.
All proposal checks passed for n=3.
All proposal checks passed for n=4.
All proposal checks passed for n=5.
All proposal checks passed for n=6.
All proposal checks passed for n=7.
All proposal checks passed for n=8.
All proposal checks passed for n=9.
All proposal checks passed for n=10.
All proposal checks passed for n=11.
All proposal checks passed for n=12.
All proposal checks passed for n=13.
All proposal checks passed for n=14.
All proposal checks passed for n=15.
All proposal checks passed for n=16.
All proposal checks passed for n=32.
All proposal checks passed for n=64.
All proposal checks passed for n=128.


<a id="fully-phase-based-arithmetic"></a>
## Fully phase-based arithmetic

The following table reproduces Table V of the paper. We use $\ell_x=\lceil\log_2x\rceil$.

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| $\mathrm{PREPARE}$ | $6n$ | $60n-56$ |
| $\mathrm{SELECT}$ | $8n$ | $0$ |
| controlled-$\mathrm{SELECT}$ | $10n$ | $6n+9$ |
| $R_0$ | $6n+2$ | $14\ell_{6n}-10$ |
| controlled-$R_0$ | $6n+3$ | $14\ell_{6n}-10$ |
| $Q_{\mathrm{ph}}$ | $8n+2$ | $120n+14\ell_{6n}-122$ |
| controlled-$Q_{\mathrm{ph}}$ | $10n+2$ | $126n+14\ell_{6n}-113$ |
| Fully phase arithmetic | $10n+2$ | $3d_{\mathrm{ph}}(126n+14\ell_{6n}-113)+3(2d_{\mathrm{ph}}+1)$ |

The validation uses a fixed test degree $d_{\mathrm{ph}}$. The resource formulas are independent of the numerical values of the SK coefficients, provided the circuit structure is unchanged.


In [ ]:
from monaqa2.qiskit.arithmetic_fully_phase import (
    PrepareDeltaHamiltonian,
    SelectDeltaHamiltonian,
    ControlledSelectDeltaHamiltonian,
    ReflectionZero,
    ControlledReflectionZero,
    QubitizedDeltaHamiltonian,
    ControlledQubitizedDeltaHamiltonian,
    FullyPhaseArithmetic,
)


def validate_fully_phase_resources(
    n: int,
    *,
    d_ph: int = 4,
    verbose: bool = False,
) -> None:
    """Validate all rows of Table V."""
    if n < 2:
        raise ValueError("Use n >= 2.")

    h = np.ones(n)
    J = np.ones((n, n)) - np.eye(n)
    beta = 1.0
    eps_ops = 1e-3
    ell_6n = ceil_log2(6 * n)

    validate_resource(PrepareDeltaHamiltonian(n, h, J), 6 * n, 60 * n - 56, verbose=verbose)
    validate_resource(SelectDeltaHamiltonian(n, h, J), 8 * n, 0, verbose=verbose)
    validate_resource(ControlledSelectDeltaHamiltonian(n, h, J), 10 * n, 6 * n + 9, verbose=verbose)
    validate_resource(ReflectionZero(6 * n), 6 * n + 2, 14 * ell_6n - 10, verbose=verbose)
    validate_resource(ControlledReflectionZero(6 * n), 6 * n + 3, 14 * ell_6n - 10, verbose=verbose)
    validate_resource(QubitizedDeltaHamiltonian(n, h, J), 8 * n + 2, 120 * n + 14 * ell_6n - 122, verbose=verbose)
    validate_resource(ControlledQubitizedDeltaHamiltonian(n, h, J), 10 * n + 2, 126 * n + 14 * ell_6n - 113, verbose=verbose)
    validate_resource(
        FullyPhaseArithmetic(
            n,
            h,
            J,
            beta=beta,
            eps_ops=eps_ops,
            degree=d_ph,
            is_mocked_construction=False,
            is_mocked_angles=True,
        ),
        expected_num_qubits=10 * n + 2,
        expected_depth=3 * d_ph * (126 * n + 14 * ell_6n - 113) + 3 * (2 * d_ph + 1),
        verbose=verbose,
    )

    print(f"All fully phase-based checks passed for n={n}.")


fully_phase_sizes = list(range(2, 16)) + [2**power for power in range(4, 8)]
for n in fully_phase_sizes:
    validate_fully_phase_resources(n)


All fully phase-based checks passed for n=2.
All fully phase-based checks passed for n=3.
All fully phase-based checks passed for n=4.
All fully phase-based checks passed for n=5.
All fully phase-based checks passed for n=6.
All fully phase-based checks passed for n=7.
All fully phase-based checks passed for n=8.
All fully phase-based checks passed for n=9.
All fully phase-based checks passed for n=10.
All fully phase-based checks passed for n=11.
All fully phase-based checks passed for n=12.
All fully phase-based checks passed for n=13.
All fully phase-based checks passed for n=14.
All fully phase-based checks passed for n=15.
All fully phase-based checks passed for n=16.
All fully phase-based checks passed for n=32.
All fully phase-based checks passed for n=64.


<a id="hybrid-arithmetic"></a>
## Hybrid fixed-point and phase arithmetic

This section validates Tables VI and VII and the complete hybrid-resource formulas in Eqs. (F15) and (F16). We use

$$
M=n+\binom{n}{2},
\qquad
w=b+1,
\qquad
s_{\mathrm{SK}}=\left\lceil\log_{3/2}(2M)\right\rceil,
\qquad
\ell_x=\left\lceil\log_2x\right\rceil.
$$

### Fixed-point part: Table VI

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Conditional terms loader | $Mw+2M-n$ | $0$ |
| Three-to-two compressor | $5$ | $9$ |
| Wallace-tree adder | $6Mw-2w-2M+1$ | $18s_{\mathrm{SK}}+18w$ |
| Energy-difference block | $2n+6Mw-2w-2M+1$ | $18s_{\mathrm{SK}}+18w$ |
| Positive-part selector | $3w$ | $3$ |
| Cutoff-tail block | $2w+3$ | $14\ell_{j_{\mathrm{cut}}}+3w-3j_{\mathrm{cut}}-13$ |

The cutoff-tail formula assumes the active-tail regime $3\leq j_{\mathrm{cut}}\leq w-1$.

### Phase-arithmetic part: Table VII

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| One-body $\mathrm{PREPARE}$ | $w$ | $2\ell_w$ |
| One-body $\mathrm{SELECT}$ | $2w$ | $0$ |
| Controlled one-body $\mathrm{SELECT}$ | $3w$ | $3$ |
| One-body reflection | $w+2$ | $14\ell_w-10$ |
| Controlled one-body reflection | $w+3$ | $14\ell_w-10$ |
| One-body qubitized operator | $2w+2$ | $18\ell_w-10$ |
| Controlled one-body qubitized operator | $3w+2$ | $18\ell_w-7$ |
| Square-root exponential arithmetic | $3w+2$ | $d_{\mathrm{hyb}}(54\ell_w-18)+3$ |

The complete hybrid block uses

$$
2n+6Mw-2M+2
$$

logical qubits and has non-Clifford depth

$$
d_{\mathrm{hyb}}(54\ell_w-18)
+
36s_{\mathrm{SK}}
+
42w
-
6j_{\mathrm{cut}}
+
28\ell_{j_{\mathrm{cut}}}
-
17.
$$


In [ ]:
from monaqa2.qiskit.arithmetic_hybrid_updated import (
    ThreeTwoCompressor,
    WallaceTreeAdder,
    ConditionalTermsLoader,
    DeltaEnergy,
    PrepareOneBodyHamiltonian,
    SelectOneBodyHamiltonian,
    ControlledSelectOneBodyHamiltonian,
    QubitizedOneBodyHamiltonian,
    ControlledQubitizedOneBodyHamiltonian,
    CutoffTail,
    SqrtExpArithmetic,
    PositivePartSelector,
    HybridPhaseArithmetic,
)


def validate_hybrid_arithmetic_resources(
    n: int,
    w: int,
    j_cut: int,
    d_hyb: int,
) -> None:
    """Validate Tables VI-VII and Eqs. (F15)-(F16)."""
    if n < 2:
        raise ValueError("Use n >= 2.")
    if not (3 <= j_cut <= w - 1):
        raise ValueError("The cutoff-tail formula requires 3 <= j_cut <= w - 1.")

    M = n * (n + 1) // 2
    b = w - 1
    s_sk = int(np.ceil(np.log(2 * M) / np.log(3.0 / 2.0)))
    ell_w = ceil_log2(w)
    ell_j_cut = ceil_log2(j_cut)

    # Coefficients chosen so that Eq. (F12) gives the requested j_cut.
    h = np.zeros(n)
    h[0] = 0.5
    J = np.zeros((n, n))
    h_signal = np.ones(w)

    Lambda = ConditionalTermsLoader._alpha(h, J)
    eps_ops = float(np.exp(-1.0))
    eps_tail = eps_ops
    beta = float(2 ** (j_cut + 1))

    computed_j_cut = max(
        0,
        min(
            w - 1,
            int(np.floor(np.log2(beta * Lambda / (2 * np.log(1 / eps_tail))))),
        ),
    )
    if computed_j_cut != j_cut:
        raise ValueError(f"Constructed parameters give j_cut={computed_j_cut}; expected {j_cut}.")

    validate_resource(ConditionalTermsLoader(n, h, J, b, invert_coefficients=False), M * w + 2 * M - n, 0)
    validate_resource(ThreeTwoCompressor(), 5, 9)
    validate_resource(WallaceTreeAdder(2 * M, w), 6 * M * w - 2 * w - 2 * M + 1, 18 * s_sk + 18 * w)
    validate_resource(DeltaEnergy(n, h, J, b), 2 * n + 6 * M * w - 2 * w - 2 * M + 1, 18 * s_sk + 18 * w)
    validate_resource(PositivePartSelector(w), 3 * w, 3)
    validate_resource(CutoffTail(w, j_cut), 2 * w + 3, 14 * ell_j_cut + 3 * w - 3 * j_cut - 13)

    validate_resource(PrepareOneBodyHamiltonian(w, h_signal), w, 2 * ell_w)
    validate_resource(SelectOneBodyHamiltonian(w, h_signal), 2 * w, 0)
    validate_resource(ControlledSelectOneBodyHamiltonian(w, h_signal), 3 * w, 3)
    validate_resource(ReflectionZero(w), w + 2, 14 * ell_w - 10)
    validate_resource(ControlledReflectionZero(w), w + 3, 14 * ell_w - 10)
    validate_resource(QubitizedOneBodyHamiltonian(w, h_signal), 2 * w + 2, 18 * ell_w - 10)
    validate_resource(ControlledQubitizedOneBodyHamiltonian(w, h_signal), 3 * w + 2, 18 * ell_w - 7)
    validate_resource(
        SqrtExpArithmetic(w, beta, Lambda, eps_ops, eps_tail=eps_tail, degree=d_hyb, is_mocked_angles=True),
        3 * w + 2,
        d_hyb * (54 * ell_w - 18) + 3,
    )

    validate_resource(
        HybridPhaseArithmetic(n, h, J, b, beta, eps_ops, degree=d_hyb, is_mocked_angles=True),
        2 * n + 6 * M * w - 2 * M + 2,
        d_hyb * (54 * ell_w - 18) + 36 * s_sk + 42 * w - 6 * j_cut + 28 * ell_j_cut - 17,
    )

    print(f"All hybrid checks passed for n={n}, M={M}, w={w}, j_cut={j_cut}, d_hyb={d_hyb}.")


hybrid_test_cases = [
    (2, 5, 3, 1),
    (3, 6, 3, 1),
    (4, 7, 3, 2),
    (5, 8, 4, 2),
    (5, 9, 4, 3),
    (6, 10, 4, 3),
    (7, 11, 5, 4),
    (8, 12, 5, 4),
    (10, 12, 5, 5),
    (12, 14, 6, 6),
    (15, 16, 7, 8),
    (6, 9, 3, 3),
    (6, 9, 8, 3),
]

for test_case in hybrid_test_cases:
    validate_hybrid_arithmetic_resources(*test_case)


<a id="reflection-and-accept-path"></a>
## Reflection and accept-path unitary

The following checks reproduce Table VIII for the hybrid Boltzmann coin. Its persistent coin register contains the GQSP control qubit and the $w$-qubit one-body selection register, so it has $w+1$ qubits.

| Component | Logical qubits | Non-Clifford depth |
|---|---:|---:|
| Reflection | $2n+w+3$ | $14\ell_{n+w}-10$ |
| Accept-path unitary | $3n+w+1$ | $28\ell_{w+1}-11$ |

The reflection conditions on the proposed-state register and the persistent coin register being zero. The accept-path unitary computes the zero-coin flag, fans it out using Clifford gates, applies the $n$ controlled swaps in parallel, and uncomputes the flag.


In [ ]:
from monaqa2.qiskit.reflection_updated import Reflection
from monaqa2.qiskit.accept_path_updated import AcceptPath


def validate_reflection_accept_path_resources(n: int, w: int) -> None:
    """Validate the two rows of Table VIII."""
    if n < 3:
        raise ValueError("Use n >= 3.")
    if w < 2:
        raise ValueError("Use w >= 2.")

    coin_qubits = w + 1
    validate_resource(
        Reflection(n, coins=coin_qubits),
        2 * n + w + 3,
        14 * ceil_log2(n + w) - 10,
    )
    validate_resource(
        AcceptPath(n, coins=coin_qubits),
        3 * n + w + 1,
        28 * ceil_log2(w + 1) - 11,
    )
    print(f"Table VIII checks passed for n={n}, w={w}.")


reflection_test_cases = [
    (3, 2),
    (3, 6),
    (4, 8),
    (8, 8),
    (8, 15),
    (16, 15),
    (16, 31),
    (32, 31),
    (32, 63),
    (64, 63),
]

for test_case in reflection_test_cases:
    validate_reflection_accept_path_resources(*test_case)
